In [6]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(".").resolve()
PROJECT_ROOT = next(
    (p for p in [ROOT, *ROOT.parents] if (p / ".git").exists() or (p / "pyproject.toml").exists()),
    ROOT
)

mwi_path = PROJECT_ROOT / "01-input" / "country" / "MWI_1997.dta"
mwi = pd.read_stata(mwi_path, convert_categoricals=False)
mwi

,countrycode,hhid,welfare,urban,hsize,wta_hh,weight,fdpindex,nfdpindex,pc_hhdr,...,comparable,cpi2005,icp2005,cpi_data_level,cpi2011_SM26,cpi2017_SM26,cpi2021_SM26,icp2011_SM26,icp2017_SM26,icp2021_SM26
0,MWI,1.017700e+10,7068.166016,0,1,187.221756,187.221756,0.741,0.741,9538.685547,...,0,NaN,NaN,2,0.117,0.031429,0.021523,78.703018,241.930527,282.094238
1,MWI,1.017700e+10,9334.137695,0,2,187.221756,374.443512,0.741,0.741,12596.676758,...,0,NaN,NaN,2,0.117,0.031429,0.021523,78.703018,241.930527,282.094238
2,MWI,1.017700e+10,6705.055176,0,4,187.221756,748.887024,0.741,0.741,9048.657227,...,0,NaN,NaN,2,0.117,0.031429,0.021523,78.703018,241.930527,282.094238
3,MWI,1.017700e+10,3693.166260,0,7,187.221756,1310.552246,0.741,0.741,4984.030273,...,0,NaN,NaN,2,0.117,0.031429,0.021523,78.703018,241.930527,282.094238
4,MWI,1.017700e+10,5462.660156,0,2,187.221756,374.443512,0.741,0.741,7372.010742,...,0,NaN,NaN,2,0.117,0.031429,0.021523,78.703018,241.930527,282.094238
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10693,MWI,3.535010e+11,6549.470703,0,5,114.045456,570.227295,1.124,1.124,5826.931641,...,0,NaN,NaN,2,0.117,0.031429,0.021523,78.703018,241.930527,282.094238
10694,MWI,3.535010e+11,4532.853027,0,7,114.045456,798.318176,1.124,1.124,4032.787598,...,0,NaN,NaN,2,0.117,0.031429,0.021523,78.703018,241.930527,282.094238
10695,MWI,3.535010e+11,16584.109375,0,1,114.045456,114.045456,1.124,1.124,14754.546875,...,0,NaN,NaN,2,0.117,0.031429,0.021523,78.703018,241.930527,282.094238
10696,MWI,3.535010e+11,13787.355469,0,2,114.045456,228.090912,1.124,1.124,12266.331055,...,0,NaN,NaN,2,0.117,0.031429,0.021523,78.703018,241.930527,282.094238


## Build 20,000 weighted bins
This block recreates a 20,000-point weighted distribution from `MWI_1997.dta` using weighted quantiles (equal-population points).

In [7]:
# Prepare microdata columns used for quantization
required_cols = ["welfare", "weight"]
missing_cols = [c for c in required_cols if c not in mwi.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

d = mwi[required_cols].dropna().copy()
d = d[d["weight"] > 0].copy()

# Optional PPP conversion if CPI/ICP columns are available
if "cpi2021" in mwi.columns and "icp2021" in mwi.columns:
    cpi = mwi["cpi2021"].dropna()
    icp = mwi["icp2021"].dropna()
    if not cpi.empty and not icp.empty:
        d["welfare"] = d["welfare"] / (float(cpi.iloc[0]) * float(icp.iloc[0]) * 365.0)

# Add reporting level for compatibility with lineup-like format
if "reporting_level" in mwi.columns:
    d["reporting_level"] = mwi.loc[d.index, "reporting_level"].astype(str).fillna("national")
else:
    d["reporting_level"] = "national"


def weighted_quantile_points(df, nobs=20000):
    probs = np.arange(1, nobs + 1, dtype=float) / nobs - 5.0 / (nobs * 10.0)
    probs = np.clip(probs, 0.0, 1.0)

    out = []
    for rl, sub in df.groupby("reporting_level", observed=True):
        s = sub.sort_values("welfare").copy()
        w = s["weight"].to_numpy(dtype=float)
        x = s["welfare"].to_numpy(dtype=float)

        wsum = w.sum()
        if wsum <= 0:
            continue

        cw = np.cumsum(w) / wsum
        qx = np.interp(probs, cw, x)

        out.append(
            pd.DataFrame(
                {
                    "reporting_level": rl,
                    "welfare": qx,
                    "weight": np.repeat(wsum / nobs, nobs),
                    "quantile_point": np.arange(1, nobs + 1, dtype=int),
                }
            )
        )

    if not out:
        raise ValueError("No valid rows to build quantile points.")

    return pd.concat(out, ignore_index=True)


bins_20000 = weighted_quantile_points(d, nobs=20000)
bins_20000.head(), bins_20000.shape

(  reporting_level   welfare      weight  quantile_point
 0        national  0.202600  483.332405               1
 1        national  0.202600  483.332405               2
 2        national  0.202600  483.332405               3
 3        national  0.214396  483.332405               4
 4        national  0.240749  483.332405               5,
 (20000, 4))

In [8]:
# Save 20,000-point distribution
out_path_20000 = PROJECT_ROOT / "target_file" / "MWI_1997_20000_bins.csv"
out_path_20000.parent.mkdir(parents=True, exist_ok=True)
bins_20000.to_csv(out_path_20000, index=False)
out_path_20000

WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/bindata_check/target_file/MWI_1997_20000_bins.csv')

In [9]:
# Weighted mean of welfare after 20k grouping
mean_welfare_20000 = (bins_20000["welfare"] * bins_20000["weight"]).sum() / bins_20000["weight"].sum()
mean_welfare_20000

np.float64(4.912689083770458)

## Belgium 2000: Build 20,000 weighted bins and mean welfare

In [10]:
# Load BEL 2000 and prepare welfare/weight input
bel_path = PROJECT_ROOT / "01-input" / "country" / "BEL_2000.dta"
if not bel_path.exists():
    raise FileNotFoundError(f"BEL_2000.dta not found at: {bel_path}")

bel = pd.read_stata(bel_path, convert_categoricals=False)

required_cols_bel = ["welfare", "weight"]
missing_cols_bel = [c for c in required_cols_bel if c not in bel.columns]
if missing_cols_bel:
    raise ValueError(f"BEL_2000 missing required columns: {missing_cols_bel}")

d_bel = bel[required_cols_bel].dropna().copy()
d_bel = d_bel[d_bel["weight"] > 0].copy()

if "cpi2021" in bel.columns and "icp2021" in bel.columns:
    cpi_bel = bel["cpi2021"].dropna()
    icp_bel = bel["icp2021"].dropna()
    if not cpi_bel.empty and not icp_bel.empty:
        d_bel["welfare"] = d_bel["welfare"] / (float(cpi_bel.iloc[0]) * float(icp_bel.iloc[0]) * 365.0)

if "reporting_level" in bel.columns:
    d_bel["reporting_level"] = bel.loc[d_bel.index, "reporting_level"].astype(str).fillna("national")
else:
    d_bel["reporting_level"] = "national"

bins_20000_bel = weighted_quantile_points(d_bel, nobs=20000)
out_path_20000_bel = PROJECT_ROOT / "target_file" / "BEL_2000_20000_bins.csv"
bins_20000_bel.to_csv(out_path_20000_bel, index=False)

bins_20000_bel.head(), bins_20000_bel.shape, out_path_20000_bel

(  reporting_level  welfare      weight  quantile_point
 0        national      0.0  366.132251               1
 1        national      0.0  366.132251               2
 2        national      0.0  366.132251               3
 3        national      0.0  366.132251               4
 4        national      0.0  366.132251               5,
 (20000, 4),
 WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/bindata_check/target_file/BEL_2000_20000_bins.csv'))

In [11]:
# Weighted mean of welfare after BEL 2000 20k grouping
mean_welfare_20000_bel = (bins_20000_bel["welfare"] * bins_20000_bel["weight"]).sum() / bins_20000_bel["weight"].sum()
mean_welfare_20000_bel

np.float64(61.20624383087987)